# Luxor9 OS — Wan 2.1 & ComfyUI Automated Video Factory (Kaggle)

**GitHub References:**
- [Wan-Video/Wan2.1](https://github.com/Wan-Video/Wan2.1)
- [Comfy-Org/ComfyUI](https://github.com/Comfy-Org/ComfyUI)

Sets up ComfyUI, downloads the **Wan2.1-T2V-1.3B-FP8** weights, and headless-renders every scene in the CONFIG cell. Outputs go to `/kaggle/working/renders` — download them from the **Output** tab (or via the Kaggle API).

**Before running — Notebook Settings (right sidebar):**
- Accelerator: **GPU T4 x2** (or P100)  ·  Internet: **ON**

**Run order:** 1 → 2 → 3 → … → 7 (or **Run All**). Edit cell 2 (CONFIG) to change scenes, resolution, steps, or output format.

In [ ]:
# 1. Check GPU
import shutil
import subprocess
import sys

if shutil.which("nvidia-smi") is None:
    sys.exit("ERROR: nvidia-smi not found — this kernel is running WITHOUT a GPU. "
             "Enable Accelerator: GPU T4 x2 in the kernel settings and re-run.")
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print(gpu.stdout.strip() or gpu.stderr.strip())
print("GPU OK — Wan 2.1 T2V 1.3B fp8 runs on a T4/P100.")


In [ ]:
# 2. CONFIG — edit scenes & render settings here
import os

# Kaggle working dir survives the session and is downloadable from the Output tab.
OUTPUT_DIR = "/kaggle/working/renders"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Bumped by automate_render.py --force to guarantee a new Kaggle run.
RENDER_VERSION = "2026-08-15.6"

DEFAULTS = {
    "width": 1280,
    "height": 720,
    "length": 81,       # frames (~3.4s at 24 fps)
    "steps": 30,
    "cfg": 6.5,
    "fps": 24,
    "sampler_name": "euler",
    "scheduler": "simple",
    "seed": None,       # None = random per scene; set a number for reproducible renders
}

OUTPUT_FORMAT = "mp4"   # "mp4" (VideoHelperSuite) or "webp" (built-in, no extra install)

SCENES = [
    {
        "prefix": "Luxor9_Scene01",
        "prompt": (
            "Cinematic 4K shot, 24fps. Symmetrical medium tracking shot of an open sovereign terminal "
            "inside a dark technical noir observatory. Multi-agent AI nodes execute zero-latency liquidity "
            "routing across floating obsidian glass interfaces, glowing 0.5px gold velocity lines, "
            "volumetric atmospheric lighting, anamorphic lens flare, shallow depth of field."
        ),
    },
    {
        "prefix": "Luxor9_Scene02",
        "prompt": (
            "Extreme close-up macro shot of gold liquidity routing lines weaving through a floating "
            "holographic trading terminal, dark noir palette, volumetric fog, glowing particle trails, "
            "cinematic lighting, 24fps, shallow depth of field."
        ),
    },
    {
        "prefix": "Luxor9_Scene03",
        "prompt": (
            "Wide establishing shot of the sovereign terminal interior: a vast dark observatory where "
            "multi-agent AI nodes hover as points of light, obsidian glass interfaces floating in layers, "
            "anamorphic lens flare, volumetric atmospheric lighting, cinematic 24fps."
        ),
    },
]

MODEL_FILES = {
    "unet": "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_t2v_1.3B_fp8_e4m3fn.safetensors",
    "unet_name": "wan2.1_t2v_1.3B_fp8_e4m3fn.safetensors",
    "clip": "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip/umt5_xxl_fp8_e4m3fn.safetensors",
    "clip_name": "umt5_xxl_fp8_e4m3fn.safetensors",
    "vae": "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors",
    "vae_name": "wan_2.1_vae.safetensors",
}

In [ ]:
# 3. Clone ComfyUI & install dependencies.
# NOTE: We do NOT reinstall torch — Kaggle's preinstalled CUDA build is used.
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI || echo "ComfyUI already present"
%cd /kaggle/working/ComfyUI
!pip install -q -r requirements.txt
!pip install -q diffusers transformers accelerate sentencepiece

In [ ]:
# 4. Download Wan 2.1 T2V 1.3B weights (fp8), text encoder, and VAE from Comfy-Org's repackaged repo.
import os
import subprocess

MODEL_DIR = "/kaggle/working/ComfyUI/models"

def download(url, filename, subdir):
    dest = os.path.join(MODEL_DIR, subdir, filename)
    if os.path.exists(dest):
        print("Already present:", dest)
        return
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    print("Downloading", filename, "...")
    subprocess.run(["wget", "-q", "-c", url, "-O", dest], check=True)
    print("OK:", dest)

download(MODEL_FILES["unet"], MODEL_FILES["unet_name"], "diffusion_models")
download(MODEL_FILES["clip"], MODEL_FILES["clip_name"], "clip")
download(MODEL_FILES["vae"], MODEL_FILES["vae_name"], "vae")

In [ ]:
# 5. Install VideoHelperSuite custom node (required for MP4 output).
!git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git /kaggle/working/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite || echo "VideoHelperSuite already present"
%cd /kaggle/working/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite
!pip install -q -r requirements.txt
%cd /kaggle/working/ComfyUI

In [ ]:
# 6. Start ComfyUI headless (readiness polling) + optional Cloudflare tunnel
import json
import subprocess
import threading
import time
import urllib.request

API = "http://127.0.0.1:8188"

comfy_proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd="/kaggle/working/ComfyUI",
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Poll until the API responds (first boot can take a while — no fixed sleep).
ready = False
for _ in range(120):
    try:
        with urllib.request.urlopen(f"{API}/system_stats", timeout=3) as resp:
            json.loads(resp.read().decode())
        ready = True
        break
    except Exception:
        time.sleep(5)

if not ready:
    raise RuntimeError("ComfyUI did not become ready on port 8188 — check the session logs.")

print("ComfyUI ready at", API)

# Optional Cloudflare quick tunnel for browser access to the WebUI.
# Wrapped in try/except — rendering works fine without it.
try:
    subprocess.run(
        ["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/kaggle/working/cloudflared"],
        check=True,
    )
    subprocess.run(["chmod", "+x", "/kaggle/working/cloudflared"], check=True)

    cf_proc = subprocess.Popen(
        ["/kaggle/working/cloudflared", "tunnel", "--url", API, "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    tunnel_url = None
    def read_cf_logs():
        global tunnel_url
        for line in cf_proc.stdout:
            line = line.strip()
            if "trycloudflare.com" in line:
                start = line.find("https://")
                tunnel_url = line[start:].strip()
                print("Tunnel URL:", tunnel_url)
            elif "ERR" in line or "error" in line.lower():
                print(line)

    threading.Thread(target=read_cf_logs, daemon=True).start()
    time.sleep(8)
    if not tunnel_url:
        print("Tunnel still starting — the URL will be printed above when ready.")
except Exception as e:
    print("Tunnel skipped (not required for rendering):", e)

In [ ]:
# 7. Headless batch render — every scene in CONFIG, saved to /kaggle/working/renders.
import json
import os
import shutil
import time
import urllib.request

COMFY_OUTPUT = "/kaggle/working/ComfyUI/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def queue_and_wait(workflow):
    data = json.dumps(workflow).encode("utf-8")
    req = urllib.request.Request(f"{API}/prompt", data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as resp:
        prompt_id = json.loads(resp.read().decode())["prompt_id"]
    while True:
        with urllib.request.urlopen(f"{API}/history/{prompt_id}") as resp:
            history = json.loads(resp.read().decode())
        if prompt_id in history:
            return history[prompt_id].get("outputs", {})
        time.sleep(5)

def collect_files(outputs):
    files = []
    for node_output in outputs.values():
        for value in node_output.values():
            if isinstance(value, list):
                for item in value:
                    if isinstance(item, dict) and "filename" in item:
                        files.append(os.path.join(COMFY_OUTPUT, item["filename"]))
    return files

# If MP4 was requested, make sure the VideoHelperSuite node actually loaded.
if OUTPUT_FORMAT == "mp4":
    try:
        with urllib.request.urlopen(f"{API}/object_info/VHS_VideoCombine", timeout=5) as resp:
            json.loads(resp.read().decode())
    except Exception:
        raise RuntimeError(
            "VHS_VideoCombine node not found — re-run cell 5 and restart the session, "
            "or set OUTPUT_FORMAT = 'webp' in cell 2."
        )

results = []
for idx, scene in enumerate(SCENES, start=1):
    params = {**DEFAULTS, **scene}
    prefix = params.pop("prefix")
    prompt = params.pop("prompt")
    print(f"\n=== Rendering scene {idx}: {prefix} ===")

    workflow = {
        "prompt": {
            "1": {"inputs": {"unet_name": MODEL_FILES["unet_name"]}, "class_type": "UNETLoader"},
            "2": {"inputs": {"text": prompt, "clip": ["3", 0]}, "class_type": "CLIPTextEncode"},
            "3": {"inputs": {"clip_name": MODEL_FILES["clip_name"], "type": "wan"}, "class_type": "CLIPLoader"},
            "4": {"inputs": {"text": "blurry, low quality, deformed, artifacts", "clip": ["3", 0]}, "class_type": "CLIPTextEncode"},
            "5": {"inputs": {
                "width": params["width"], "height": params["height"],
                "length": params["length"], "batch_size": 1,
            }, "class_type": "WanEmptyLatentVideo"},
            # model_type="wan" is required for Wan models in current ComfyUI.
            "6": {"inputs": {
                "seed": params["seed"] if params["seed"] is not None else int(time.time()) + idx,
                "steps": params["steps"],
                "cfg": params["cfg"],
                "sampler_name": params["sampler_name"],
                "scheduler": params["scheduler"],
                "model_type": "wan",
                "positive": ["2", 0],
                "negative": ["4", 0],
                "model": ["1", 0],
                "latent_image": ["5", 0],
            }, "class_type": "KSampler"},
            "7": {"inputs": {"vae_name": MODEL_FILES["vae_name"]}, "class_type": "VAELoader"},
            "8": {"inputs": {"samples": ["6", 0], "vae": ["7", 0]}, "class_type": "VAEDecode"},
        }
    }

    if OUTPUT_FORMAT == "mp4":
        workflow["prompt"]["9"] = {"inputs": {
            "images": ["8", 0],
            "frame_rate": float(params["fps"]),
            "loop_count": 0,
            "filename_prefix": prefix,
            "format": "video/h264-mp4",
            "pix_fmt": "yuv420p",
            "fps": params["fps"],
            "codec": "libx264",
            "quality": 1,
            "pix_fmt_in_image": "rgb24",
        }, "class_type": "VHS_VideoCombine"}
    else:
        workflow["prompt"]["9"] = {"inputs": {
            "filename_prefix": prefix,
            "fps": params["fps"],
            "images": ["8", 0],
        }, "class_type": "SaveAnimatedWEBP"}

    outputs = queue_and_wait(workflow)
    files = collect_files(outputs)
    if not files:
        print("  No output files found. Check the ComfyUI console for errors.")
        continue
    for src in files:
        dst = os.path.join(OUTPUT_DIR, os.path.basename(src))
        shutil.copy2(src, dst)
        print("  Saved:", dst)
        results.append(dst)

print(f"\nDone — {len(results)} file(s) in /kaggle/working/renders")
print("Download them from the Output tab, or run locally:")
print("  kaggle kernels output <username>/<slug> -p ./renders")

## Notes & troubleshooting

- **Settings:** Accelerator **GPU T4 x2** (or P100), Internet **ON** — set before running.
- **Config:** edit cell 2 — add scenes, override per-scene `steps`/`cfg`/`seed`/`width`/`height`/`length`, or switch `OUTPUT_FORMAT` between `"mp4"` and `"webp"`.
- **Model:** Wan 2.1 T2V 1.3B fp8 — fits a T4/P100. The 14B variant will OOM.
- **Output:** every render lands in `/kaggle/working/renders` → download from the **Output** tab, or with `kaggle kernels output <username>/<slug> -p .`.
- **No Drive on Kaggle:** to persist renders long-term, download them after the run (Kaggle sessions are deleted after ~9h; GPU quota ~30h/week).
- **If a cell errors:** re-run from the failed cell. Clones and downloads are idempotent (`|| echo already present`, `wget -c`).
- **Tunnel:** optional; if it fails, rendering still proceeds.